# IMDB RNN Keras 3 Walkthrough

This notebook is the interactive path through the recurrent neural networks chapter experiment. Use it to inspect token sequences, padding, masks, model summaries, validation curves, and controlled comparisons one cell at a time.

The source-of-truth training code is still `imdb_rnn_keras3.py` in this directory. The notebook imports that script and reuses its data-loading, model-building, callback, metric, and artifact helpers. For final homework comparisons, use the shared raw-text IMDB data from the embeddings chapter.

## Reader Preflight

Before running this notebook as public companion code:

- Start with the dependency check or smoke path. Real training, generation, or evaluation cells are usually guarded by flags such as `RUN_* = False`.
- Confirm dataset and model access, license or terms, and local paths before enabling external downloads or long runs.
- Keep secrets out of notebook cells. If a token is required, load it from the environment or `.env`, and keep `.env` secrets-only.
- Treat printed paths and saved JSON, CSV, and PNG artifacts as the evidence record. Rerun from a clean kernel before reporting results.

## 1. Select the Backend and Import the Companion Script

Choose the Keras backend before importing Keras. If you change the backend after running this cell, restart the kernel before running the notebook again.

In [ ]:
from pathlib import Path
from types import SimpleNamespace
import json
import os
import platform
import sys
import time

# External KERAS_BACKEND wins. Default is TensorFlow for convenience.
os.environ.setdefault("KERAS_BACKEND", "tensorflow")


def find_code_dir() -> Path:
    script_name = "imdb_rnn_keras3.py"
    chapter_name = "chapter_recurrent_neural_networks"
    for base in [Path.cwd(), *Path.cwd().parents]:
        for candidate in (base, base / chapter_name, base / "code" / chapter_name):
            if (candidate / script_name).exists():
                return candidate.resolve()
    raise FileNotFoundError("Run this notebook from the chapter code directory or the repository root.")


CODE_DIR = find_code_dir()
sys.path.insert(0, str(CODE_DIR))
import imdb_rnn_keras3 as rnn

keras, layers = rnn.import_keras(
    SimpleNamespace(backend=os.environ["KERAS_BACKEND"])
)

print("Python:", platform.python_version())
print("Keras:", keras.__version__)
print("Backend:", keras.backend.backend())
print("Code directory:", CODE_DIR)

## 2. Set a Small Reproducible Run Configuration

The defaults below are intentionally bounded for notebook iteration. They are useful for checking the pipeline and comparing models, but a final report should use the full shared IMDB data unless the assignment says otherwise.

In [ ]:
CONFIG = SimpleNamespace(
    backend=os.environ["KERAS_BACKEND"],
    model="gru",
    epochs=2,
    batch_size=128,
    seed=1234,
    data_source="shared-imdb",
    data_dir=str(CODE_DIR.parent / "chapter_embeddings" / "data"),
    num_words=10000,
    max_length=200,
    embedding_dim=64,
    hidden_size=64,
    dropout=0.0,
    recurrent_dropout=0.0,
    clipnorm=1.0,
    early_stopping_patience=1,
    early_stopping_min_delta=0.0,
    validation_size=5000,
    limit_train=4000,
    limit_val=1000,
    limit_test=1000,
    quick=False,
    synthetic_data=False,
    evaluate_test=False,
    artifact_dir="artifacts/notebook",
    save_artifacts=False,
    check_deps=False,
    allow_missing_deps=False,
)

keras.utils.set_random_seed(CONFIG.seed)
print(json.dumps(vars(CONFIG), indent=2))

## 3. Load Shared IMDB, Pad Sequences, and Inspect Lengths

The script reserves token ID `0` for padding and token ID `1` for out-of-vocabulary words. The padding ratios below are part of the experiment record because wasted padded positions affect runtime.

In [ ]:
if CONFIG.synthetic_data:
    dataset_source = "synthetic"
    (x_train, y_train), (x_val, y_val), (x_test, y_test), length_summary = (
        rnn.make_synthetic_data(CONFIG)
    )
elif CONFIG.data_source == "keras-imdb":
    dataset_source = "keras_imdb"
    (x_train, y_train), (x_val, y_val), (x_test, y_test), length_summary = (
        rnn.load_imdb_data(CONFIG, keras)
    )
else:
    dataset_source = "shared_imdb"
    (x_train, y_train), (x_val, y_val), (x_test, y_test), length_summary = (
        rnn.load_shared_imdb_data(CONFIG)
    )

print("dataset_source:", dataset_source)
print("train:", x_train.shape, y_train.shape)
print("val:", x_val.shape, y_val.shape)
print("test:", x_test.shape, y_test.shape)
print(f"train_padding_ratio={rnn.padding_ratio(x_train):.4f}")
print(f"val_padding_ratio={rnn.padding_ratio(x_val):.4f}")
print(f"test_padding_ratio={rnn.padding_ratio(x_test):.4f}")
print("mean_raw_lengths:", length_summary)

## 4. Decode One Review

This cell is for orientation only. The model receives integer token IDs, not strings.

In [ ]:
if dataset_source == "keras_imdb":
    word_index = keras.datasets.imdb.get_word_index()
    reverse_word_index = {index + 3: word for word, index in word_index.items()}
    reverse_word_index.update({0: "<PAD>", 1: "<START>", 2: "<UNK>", 3: "<UNUSED>"})

    def decode_review(row, limit=80):
        tokens = [int(token) for token in row if int(token) != 0][:limit]
        return " ".join(reverse_word_index.get(token, "?") for token in tokens)

    print("label:", int(y_train[0]))
    print(decode_review(x_train[0]))
elif dataset_source == "shared_imdb":
    print("Shared raw-text examples use the chapter tokenizer and vocabulary built from the training split.")
    print("First non-padding token IDs:", [int(token) for token in x_train[0] if int(token) != 0][:40])
else:
    print("Synthetic examples are generated token IDs, so there is no word index to decode.")

## 5. Check the Padding Mask

`mask_zero=True` makes the embedding layer produce a Boolean mask where `False` marks padded positions. A recurrent layer that supports masks can then ignore those positions when selecting the final state.

In [ ]:
mask_probe = layers.Embedding(
    input_dim=CONFIG.num_words,
    output_dim=CONFIG.embedding_dim,
    mask_zero=True,
)
mask = mask_probe.compute_mask(x_train[:2])
mask_array = keras.ops.convert_to_numpy(mask)

print("mask shape:", mask_array.shape)
print("first 40 mask values for example 0:", mask_array[0, :40])
print("valid token counts:", mask_array.sum(axis=1))

## 6. Define a Controlled Training Helper

The helper changes only the model family. Data split, vocabulary, maximum length, batch size, optimizer, clipping, and epoch budget stay fixed.

In [ ]:
def run_args(model_name, **overrides):
    values = vars(CONFIG).copy()
    values.update(overrides)
    values["model"] = model_name
    return SimpleNamespace(**values)


def train_one(model_name):
    args = run_args(model_name)
    keras.utils.set_random_seed(args.seed)
    model = rnn.build_model(args, keras, layers)
    print(f"\n=== {model_name} ===")
    model.summary()

    start = time.perf_counter()
    history = model.fit(
        x_train,
        y_train,
        validation_data=(x_val, y_val),
        batch_size=args.batch_size,
        epochs=args.epochs,
        callbacks=rnn.make_callbacks(args, keras),
        verbose=2,
    )
    elapsed = time.perf_counter() - start

    best_val_loss, best_val_loss_epoch = rnn.best_metric(
        history.history["val_loss"], maximize=False
    )
    best_val_accuracy, best_val_accuracy_epoch = rnn.best_metric(
        history.history["val_accuracy"], maximize=True
    )
    record = {
        "notebook_cell": "controlled training helper",
        "model": model_name,
        "dataset_source": dataset_source,
        "backend": keras.backend.backend(),
        "seed": args.seed,
        "num_words": args.num_words,
        "max_length": args.max_length,
        "embedding_dim": args.embedding_dim,
        "hidden_size": args.hidden_size,
        "dropout": args.dropout,
        "recurrent_dropout": args.recurrent_dropout,
        "clipnorm": args.clipnorm,
        "batch_size": args.batch_size,
        "epochs_requested": args.epochs,
        "epochs_completed": len(history.history["val_loss"]),
        "params": int(model.count_params()),
        "train_padding_ratio": rnn.padding_ratio(x_train),
        "val_padding_ratio": rnn.padding_ratio(x_val),
        "elapsed_seconds": elapsed,
        "best_val_loss": best_val_loss,
        "best_val_loss_epoch": best_val_loss_epoch,
        "best_val_accuracy": best_val_accuracy,
        "best_val_accuracy_epoch": best_val_accuracy_epoch,
        "evaluate_test": False,
    }
    return {"model": model, "history": history, "record": record}

### Reader Checkpoint

After the previous cell runs, confirm that the printed paths, shapes, commands, or tables match the section description before moving on. If this checkpoint fails in a public-repo environment, fix dependencies, data paths, or guarded flags before starting longer runs.

## 7. Run a Baseline and One Recurrent Model

Start with `average` versus `gru`. Add `simple-rnn` or `lstm` after the first comparison runs cleanly.

In [ ]:
RUN_MODELS = ["average", "gru"]

runs = {model_name: train_one(model_name) for model_name in RUN_MODELS}
records = [runs[model_name]["record"] for model_name in RUN_MODELS]

for record in records:
    print(
        f"{record['model']:>10} | "
        f"params={record['params']} | "
        f"best_val_accuracy={record['best_val_accuracy']:.4f} | "
        f"elapsed_seconds={record['elapsed_seconds']:.2f} | "
        f"test={record['evaluate_test']}"
    )

## 8. Plot Training Curves with Plotnine

Curves help reveal overfitting, unstable validation loss, or a comparison that needs more epochs.

In [ ]:
try:
    import pandas as pd
    from plotnine import aes, facet_wrap, geom_line, geom_point, ggplot, labs, theme_minimal
except ImportError:
    print("Install plotnine, or run `poetry install --with figures`, to plot curves.")
else:
    records = []
    for model_name, run in runs.items():
        history = run["history"].history
        epochs = range(1, len(history["loss"]) + 1)
        for panel, split, values in [
            ("Loss", "train", history["loss"]),
            ("Loss", "validation", history["val_loss"]),
            ("Accuracy", "train", history["accuracy"]),
            ("Accuracy", "validation", history["val_accuracy"]),
        ]:
            for epoch, value in zip(epochs, values):
                records.append(
                    {
                        "epoch": epoch,
                        "panel": panel,
                        "series": f"{model_name} {split}",
                        "value": value,
                    }
                )
    data = pd.DataFrame.from_records(records)
    data["panel"] = pd.Categorical(data["panel"], categories=["Loss", "Accuracy"], ordered=True)
    (
        ggplot(data, aes("epoch", "value", color="series", group="series"))
        + geom_line(size=0.8)
        + geom_point(size=2.0)
        + facet_wrap("~panel", scales="free_y", nrow=1)
        + labs(x="epoch", y="metric value", color="series")
        + theme_minimal()
    )


## 9. Save Notebook Artifacts When Needed

Leave `SAVE_ARTIFACTS` off for casual exploration. Turn it on when you want CSV/JSON evidence similar to the command-line script.

In [ ]:
SAVE_ARTIFACTS = False

if SAVE_ARTIFACTS:
    for model_name, run in runs.items():
        args = run_args(
            model_name,
            artifact_dir=str(Path(CONFIG.artifact_dir) / model_name),
            save_artifacts=True,
        )
        rnn.save_artifacts(args, run["history"], run["record"])
else:
    print("artifact_save=skipped")

## 10. Evaluate the Test Set Once After Model Selection

Keep `RUN_FINAL_TEST` false until the validation comparison has selected one model. Then evaluate only that selected model.

In [ ]:
SELECTED_MODEL = "gru"
RUN_FINAL_TEST = False

if RUN_FINAL_TEST:
    selected = runs[SELECTED_MODEL]["model"]
    test_loss, test_accuracy = selected.evaluate(x_test, y_test, verbose=0)
    runs[SELECTED_MODEL]["record"].update(
        {
            "evaluate_test": True,
            "test_loss": float(test_loss),
            "test_accuracy": float(test_accuracy),
        }
    )
    print(json.dumps(runs[SELECTED_MODEL]["record"], indent=2))
else:
    print("test_evaluation=skipped")

## 11. Use the Script for Full Reproducible Runs

For final evidence, run the command-line script from this directory so every run has a command, CSV history, JSON summary, and stable artifact path:

```sh
python imdb_rnn_keras3.py --model gru --epochs 5 --early-stopping-patience 2 --batch-size 128 --max-length 200 --num-words 10000 --save-artifacts
```

After selecting the final model by validation evidence, add `--evaluate-test` once.

## Reader Report Checklist

Before using results from this notebook in a report or downstream example, record:

- The notebook name, companion script, package versions, device, seed, and any guarded flags you enabled.
- The dataset or sample-data source, license or terms, split policy, and any preprocessing or synthetic fallback used.
- The exact artifact files that support the result, preferably saved JSON, CSV, or PNG files rather than transient cell output.
- The validation evidence used for model or configuration selection, and whether any final-test cell was run exactly once.
- The main limitation of the run, such as tiny synthetic data, missing optional dependencies, short training budget, or unavailable model access.